# 2.1 

### 题目：一阶马尔可夫模型+拉普拉斯平滑的条件概率估计
**题目描述**：给定字符序列 `"ababc"`，采用一阶马尔可夫模型，使用拉普拉斯（加1）平滑估计条件概率 $p(\text{'a'} \mid \text{'b'})$ 和 $p(\text{'c'} \mid \text{'b'})$，词汇表为 $\{\text{'a'},\text{'b'},\text{'c'}\}$。

**解答**：

#### 1. 提取转移对，统计频次
序列 `"ababc"` 共包含 4 个一阶转移对（前字符→后字符）：
$$a\to b,\quad b\to a,\quad a\to b,\quad b\to c$$

以前字符 `b` 为条件的转移统计：
- $count(b \to a) = 1$
- $count(b \to b) = 0$
- $count(b \to c) = 1$
- 前字符 `b` 出现的总次数：$count(b) = 1+0+1 = 2$
- 词汇表大小 $V = 3$（共a、b、c三个字符）

---

#### 2. 拉普拉斯平滑公式
加1平滑后的条件概率公式为：
$$
p(x_t \mid x_{t-1}) = \frac{count(x_{t-1} \to x_t) + 1}{count(x_{t-1}) + V}
$$
其中分子对每个转移计数加1，分母加上词汇表大小，保证所有可能转移的概率和为1。

---

#### 3. 计算 $p(\text{'a'} \mid \text{'b'})$
$$
p(\text{a} \mid \text{b}) = \frac{count(b \to a) + 1}{count(b) + V} = \frac{1 + 1}{2 + 3} = \frac{2}{5} = \boldsymbol{0.4}
$$

---

#### 4. 计算 $p(\text{'c'} \mid \text{'b'})$
$$
p(\text{c} \mid \text{b}) = \frac{count(b \to c) + 1}{count(b) + V} = \frac{1 + 1}{2 + 3} = \frac{2}{5} = \boldsymbol{0.4}
$$



In [14]:
#2.2
import re
from collections import Counter

def preprocess_text(text, n):
    """
    文本预处理函数：完成清洗、分词、构建词汇表、滑动窗口生成样本
    
    参数:
        text: 输入原始文本字符串
        n: 滑动窗口的特征序列长度
    
    返回:
        word2idx: 词汇表字典（键为单词，值为整数ID，按词频降序分配）
        (features, labels): 特征序列列表和对应标签列表
    """
    # 1. 文本转小写，去除标点符号，仅保留字母和空格
    text_lower = text.lower()
    # 正则匹配所有非小写字母、非空白的字符，替换为空
    text_clean = re.sub(r'[^a-z\s]', '', text_lower)

    # 2. 按空格分词（自动处理多空格、首尾空格）
    words = text_clean.split()

    # 3. 构建词汇表：按出现频率降序排序，频率相同按字母序稳定排序，ID从0开始分配
    word_counts = Counter(words)
    # 排序规则：优先按词频降序，次按单词字母升序
    sorted_word_items = sorted(word_counts.items(), key=lambda x: (-x[1], x[0]))
    word2idx = {word: idx for idx, (word, _) in enumerate(sorted_word_items)}

    # 4. 滑动窗口生成特征与标签，无后续词则直接忽略
    features = []
    labels = []
    # 仅保留存在对应标签的窗口：i + n < 单词总数
    for i in range(len(words) - n):
        feature_seq = words[i:i + n]
        next_word = words[i + n]
        features.append(feature_seq)
        labels.append(next_word)

    return word2idx, (features, labels)


# ------------------- 测试代码 -------------------
if __name__ == "__main__":
    # 测试1：题目示例
    print("测试1：题目示例")
    test_text1 = "The time machine"
    n1 = 2
    vocab1, (feats1, labs1) = preprocess_text(test_text1, n1)
    print("词汇表（word: ID）：", vocab1)
    print("特征列表：", feats1)
    print("标签列表：", labs1)

    # 测试2：带标点的复杂文本
    print("\n测试2：带标点的文本")
    test_text2 = "Hello, world! Hello deep learning."
    n2 = 2
    vocab2, (feats2, labs2) = preprocess_text(test_text2, n2)
    print("词汇表（word: ID）：", vocab2)
    print("特征列表：", feats2)
    print("标签列表：", labs2)

测试1：题目示例
词汇表（word: ID）： {'machine': 0, 'the': 1, 'time': 2}
特征列表： [['the', 'time']]
标签列表： ['machine']

测试2：带标点的文本
词汇表（word: ID）： {'hello': 0, 'deep': 1, 'learning': 2, 'world': 3}
特征列表： [['hello', 'world'], ['world', 'hello'], ['hello', 'deep']]
标签列表： ['hello', 'deep', 'learning']


### 3.1
---
#### 一、已知条件
无偏置线性RNN的定义：
- 隐藏状态更新：$\boldsymbol{h}_t = \boldsymbol{W}_{hh} \boldsymbol{h}_{t-1} + \boldsymbol{W}_{hx} \boldsymbol{x}_t$
- 输出计算：$\boldsymbol{o}_t = \boldsymbol{W}_{oh} \boldsymbol{h}_t$
- 总平方损失：$L = \frac{1}{2}\sum_{t=1}^T \|\boldsymbol{o}_t - \boldsymbol{y}_t\|^2$，其中$T$为序列总时间步长。

目标：通过沿时间反向传播（BPTT）推导损失$L$对权重$\boldsymbol{W}_{hh}$的梯度表达式，并说明梯度消失/爆炸的条件。

---
#### 二、梯度推导（BPTT 沿时间反向传播）
总损失为各时间步损失之和，因此梯度满足可加性：$\frac{\partial L}{\partial \boldsymbol{W}_{hh}} = \sum_{t=1}^T \frac{\partial L_t}{\partial \boldsymbol{W}_{hh}}$，其中单步损失$L_t = \frac{1}{2}\|\boldsymbol{o}_t - \boldsymbol{y}_t\|^2$。

##### 步骤1：输出层梯度
对第$t$步的损失关于输出求导：
$$\frac{\partial L_t}{\partial \boldsymbol{o}_t} = \boldsymbol{o}_t - \boldsymbol{y}_t$$

##### 步骤2：隐藏层末端梯度
由$\boldsymbol{o}_t = \boldsymbol{W}_{oh} \boldsymbol{h}_t$，根据链式法则，单步损失对第$t$步隐藏状态的梯度为：
$$\frac{\partial L_t}{\partial \boldsymbol{h}_t} = \boldsymbol{W}_{oh}^\top (\boldsymbol{o}_t - \boldsymbol{y}_t)$$

##### 步骤3：隐藏状态的反向梯度递推
隐藏状态满足递推关系$\boldsymbol{h}_{k+1} = \boldsymbol{W}_{hh} \boldsymbol{h}_k + \boldsymbol{W}_{hx} \boldsymbol{x}_{k+1}$，因此$\frac{\partial \boldsymbol{h}_{k+1}}{\partial \boldsymbol{h}_k} = \boldsymbol{W}_{hh}$。

对于第$t$步的损失，反向传播到第$k$步（$k < t$）的隐藏状态梯度为：
$$\frac{\partial L_t}{\partial \boldsymbol{h}_k} = \left( \frac{\partial \boldsymbol{h}_{k+1}}{\partial \boldsymbol{h}_k} \right)^\top \frac{\partial L_t}{\partial \boldsymbol{h}_{k+1}} = \boldsymbol{W}_{hh}^\top \cdot \frac{\partial L_t}{\partial \boldsymbol{h}_{k+1}}$$

递推展开可得间隔$t-k$步后的梯度：
$$\frac{\partial L_t}{\partial \boldsymbol{h}_k} = \left( \boldsymbol{W}_{hh}^\top \right)^{t-k} \cdot \boldsymbol{W}_{oh}^\top (\boldsymbol{o}_t - \boldsymbol{y}_t)$$

##### 步骤4：单步损失对$\boldsymbol{W}_{hh}$的梯度
由$\boldsymbol{h}_k = \boldsymbol{W}_{hh} \boldsymbol{h}_{k-1} + \dots$，根据矩阵求导的外积规则，第$k$步隐藏状态对$\boldsymbol{W}_{hh}$的梯度贡献为$\frac{\partial L_t}{\partial \boldsymbol{h}_k} \cdot \boldsymbol{h}_{k-1}^\top$。
因此单步损失$L_t$对$\boldsymbol{W}_{hh}$的总梯度为：
$$\frac{\partial L_t}{\partial \boldsymbol{W}_{hh}} = \sum_{k=1}^t \frac{\partial L_t}{\partial \boldsymbol{h}_k} \cdot \boldsymbol{h}_{k-1}^\top$$

##### 步骤5：总损失的最终梯度
引入**隐藏层误差项**$\boldsymbol{\delta}_t = \frac{\partial L}{\partial \boldsymbol{h}_t}$，表示总损失对第$t$步隐藏状态的梯度，其反向递推公式为：
1.  最后一步初始值：$\boldsymbol{\delta}_T = \boldsymbol{W}_{oh}^\top (\boldsymbol{o}_T - \boldsymbol{y}_T)$
2.  反向递推（$t = T-1, T-2, \dots, 1$）：
    $$\boldsymbol{\delta}_t = \boldsymbol{W}_{hh}^\top \boldsymbol{\delta}_{t+1} + \boldsymbol{W}_{oh}^\top (\boldsymbol{o}_t - \boldsymbol{y}_t)$$

最终，总损失对$\boldsymbol{W}_{hh}$的梯度为所有时间步误差项与前一时刻隐藏状态的外积之和：
$$\boldsymbol{\frac{\partial L}{\partial W_{hh}} = \sum_{t=1}^T \boldsymbol{\delta}_t \cdot \boldsymbol{h}_{t-1}^\top}$$

---
#### 三、梯度消失与梯度爆炸的条件
梯度在时间维度反向传递时，每经过一个时间步就会左乘一次$\boldsymbol{W}_{hh}^\top$，梯度的幅值变化由$\boldsymbol{W}_{hh}$的**谱半径**（矩阵所有特征值的模的最大值，记为$\rho(\boldsymbol{W}_{hh})$）决定：

1.  **梯度消失的条件**
    当$\rho(\boldsymbol{W}_{hh}) < 1$时，随着反向传播时间间隔的增大，$\left( \boldsymbol{W}_{hh}^\top \right)^\tau$的元素会呈指数级衰减。距离输出层越远的早期时间步，梯度会不断缩小直至趋近于0，模型无法学习长距离依赖关系，即发生**梯度消失**。

2.  **梯度爆炸的条件**
    当$\rho(\boldsymbol{W}_{hh}) > 1$时，梯度会随着反向传播步长的增加呈指数级增长，导致梯度过大、参数更新剧烈震荡，训练过程不稳定，即发生**梯度爆炸**。


In [13]:
import torch

def rnn_cell_forward(x_t, h_prev, W_hx, W_hh, b_h):
    """
    RNN单元单步前向传播（tanh激活函数）
    
    参数:
        x_t: 当前时刻输入，形状 (batch_size, input_size)
        h_prev: 上一时刻隐藏状态，形状 (batch_size, hidden_size)
        W_hx: 输入-隐藏权重，形状 (hidden_size, input_size)
        W_hh: 隐藏-隐藏权重，形状 (hidden_size, hidden_size)
        b_h: 隐藏层偏置，形状 (hidden_size,)
    
    返回:
        h_t: 当前时刻隐藏状态，形状 (batch_size, hidden_size)
        cache: 反向传播所需的缓存变量
    """
    # 1. 计算线性部分
    z = x_t @ W_hx.T + h_prev @ W_hh.T + b_h.unsqueeze(0)
    # 2. tanh激活得到当前隐藏状态
    h_t = torch.tanh(z)
    
    # 缓存反向传播需要的变量
    cache = (x_t, h_prev, h_t, W_hx, W_hh)
    return h_t, cache


def rnn_cell_backward(dh_next, cache):
    """
    RNN单元单步反向传播，计算所有参数与输入的梯度
    
    参数:
        dh_next: 损失对当前时刻隐藏状态h_t的梯度，形状 (batch_size, hidden_size)
        cache: 前向传播缓存的中间变量
    
    返回:
        dx_t: 损失对当前输入x_t的梯度，形状 (batch_size, input_size)
        dh_prev: 损失对上一时刻隐藏状态h_prev的梯度，形状 (batch_size, hidden_size)
        dW_hx: 损失对权重W_hx的梯度，形状 (hidden_size, input_size)
        dW_hh: 损失对权重W_hh的梯度，形状 (hidden_size, hidden_size)
        db_h: 损失对偏置b_h的梯度，形状 (hidden_size,)
    """
    x_t, h_prev, h_t, W_hx, W_hh = cache
    
    # 1. tanh的导数：d(tanh(z))/dz = 1 - tanh(z)^2 = 1 - h_t^2
    dz = dh_next * (1 - h_t ** 2)  # 逐元素相乘
    
    # 2. 偏置的梯度：batch维度求和
    db_h = dz.sum(dim=0)
    
    # 3. 权重W_hx的梯度
    dW_hx = dz.T @ x_t
    
    # 4. 权重W_hh的梯度
    dW_hh = dz.T @ h_prev
    
    # 5. 输入x_t的梯度
    dx_t = dz @ W_hx
    
    # 6. 上一时刻隐藏状态的梯度
    dh_prev = dz @ W_hh
    
    return dx_t, dh_prev, dW_hx, dW_hh, db_h


# ------------------- 梯度正确性验证 -------------------
if __name__ == "__main__":
    # 超参数设置
    batch_size = 4
    input_size = 8
    hidden_size = 16
    
    # 随机初始化输入与参数
    x_t = torch.randn(batch_size, input_size, requires_grad=True)
    h_prev = torch.randn(batch_size, hidden_size, requires_grad=True)
    W_hx = torch.randn(hidden_size, input_size, requires_grad=True)
    W_hh = torch.randn(hidden_size, hidden_size, requires_grad=True)
    b_h = torch.randn(hidden_size, requires_grad=True)
    
    # 1. 手动实现前向+反向
    h_t_manual, cache = rnn_cell_forward(x_t, h_prev, W_hx, W_hh, b_h)
    # 构造一个简单的损失：所有元素求和，方便验证梯度
    loss_manual = h_t_manual.sum()
    dh_next = torch.ones_like(h_t_manual)  # 损失对h_t的梯度为全1
    dx_manual, dh_prev_manual, dW_hx_manual, dW_hh_manual, db_h_manual = rnn_cell_backward(dh_next, cache)
    
    # 2. PyTorch自动求导验证
    h_t_auto = torch.tanh(x_t @ W_hx.T + h_prev @ W_hh.T + b_h)
    loss_auto = h_t_auto.sum()
    loss_auto.backward()
    
    # 3. 对比梯度（误差应接近0）
    print("梯度正确性验证")
    print(f"dx_t 误差: {torch.max(torch.abs(x_t.grad - dx_manual)).item():.8f}")
    print(f"dh_prev 误差: {torch.max(torch.abs(h_prev.grad - dh_prev_manual)).item():.8f}")
    print(f"dW_hx 误差: {torch.max(torch.abs(W_hx.grad - dW_hx_manual)).item():.8f}")
    print(f"dW_hh 误差: {torch.max(torch.abs(W_hh.grad - dW_hh_manual)).item():.8f}")
    print(f"db_h 误差: {torch.max(torch.abs(b_h.grad - db_h_manual)).item():.8f}")

梯度正确性验证
dx_t 误差: 0.00000000
dh_prev 误差: 0.00000000
dW_hx 误差: 0.00000000
dW_hh 误差: 0.00000000
db_h 误差: 0.00000000


### 4.1
---
#### 一、单个单向RNN单元的参数构成
对于输入维度为$I$、隐藏维度为$H$的标准RNN单元（包含权重与偏置），参数分为三部分：
- 隐藏状态自连权重$W_{hh}$：尺寸$H \times H$，共$H^2$个参数
- 输入-隐藏权重$W_{hx}$：尺寸$H \times I$，共$H \cdot I$个参数
- 隐藏层偏置$b_h$：尺寸$H$，共$H$个参数

单个单向RNN单元的参数总数：
$$\text{Params}_{\text{单向}} = H^2 + H \cdot I + H = H(H + I + 1)$$

双向RNN包含**前向、反向两个独立的RNN单元**，因此单层双向的参数为单向的2倍。

---
#### 二、分层计算总参数
模型共$L$层双向RNN + 1层全连接输出层，分层计算如下：

##### 1. 第1层（输入层）
输入为原始特征，维度$D$，即$I=D$。
单层双向参数：
$$
\text{Params}_{\text{层1}} = 2 \cdot H(H + D + 1) = 2H^2 + 2HD + 2H
$$

##### 2. 第2层 ~ 第$L$层（共$L-1$层中间层）
深度双向RNN中，上一层的输出为前向、反向隐藏状态的拼接，维度为$2H$，因此当前层输入维度$I=2H$。
单层双向参数：
$$
\text{Params}_{\text{中间层}} = 2 \cdot H(H + 2H + 1) = 6H^2 + 2H
$$
$L-1$层的总参数：
$$
\text{Params}_{\text{中间层总}} = (L-1) \cdot (6H^2 + 2H)
$$

##### 3. 最终输出层
全连接输出层的输入为最后一层双向RNN的拼接隐藏状态（维度$2H$），输出维度为$O$，包含权重和偏置：
$$
\text{Params}_{\text{输出层}} = O \cdot 2H + O = O(2H + 1)
$$

---
#### 三、总参数最终表达式
将三部分相加，得到整个深度双向RNN模型的参数总数：

$$
\boldsymbol{
\text{Total Params} = 2H^2 + 2HD + 2H + (L-1)(6H^2 + 2H) + O(2H + 1)
}
$$

展开整理后的形式：
$$
\text{Total Params} = (6L-4)H^2 + 2HD + 2LH + 2OH + O
$$

In [12]:
#4.2
import torch
import torch.nn as nn

class BiRNNEncoder(nn.Module):
    """
    双向RNN序列编码器
    输入形状: (seq_len, batch_size, input_dim)
    """
    def __init__(self, input_dim, hidden_dim):
        super(BiRNNEncoder, self).__init__()
        self.hidden_dim = hidden_dim
        # 定义双向RNN，batch_first=False（输入seq_len在前），tanh激活
        self.rnn = nn.RNN(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            bidirectional=True,
            batch_first=False,
            nonlinearity='tanh'
        )

    def forward(self, x):
        """
        参数:
            x: 输入序列，形状 (seq_len, batch_size, input_dim)
        返回:
            output: 每个时间步拼接后的前向+反向隐藏状态，形状 (seq_len, batch_size, 2*hidden_dim)
            final_hidden: 最终时间步拼接的序列表示，形状 (batch_size, 2*hidden_dim)
        """
        # 前向传播，output为所有时间步输出，h_n为最后一步隐藏状态
        output, h_n = self.rnn(x)
        
        # h_n形状: (num_directions, batch_size, hidden_dim) = (2, batch, hidden_dim)
        # h_n[0] 是前向RNN的最后一步状态，h_n[1] 是反向RNN的最后一步状态
        final_hidden = torch.cat((h_n[0], h_n[1]), dim=-1)
        
        return output, final_hidden


# ------------------- 功能与形状验证 -------------------
if __name__ == "__main__":
    # 超参数设置
    seq_len = 10    # 序列长度
    batch_size = 4  # 批次大小
    input_dim = 8   # 输入特征维度
    hidden_dim = 16 # 隐藏层维度

    # 随机生成输入序列
    x = torch.randn(seq_len, batch_size, input_dim)

    # 初始化编码器并前向传播
    encoder = BiRNNEncoder(input_dim, hidden_dim)
    output, final_hidden = encoder(x)

    print("形状验证")
    print(f"输入序列形状: {x.shape}")
    print(f"每个时间步拼接隐藏状态形状: {output.shape}")
    print(f"最终序列表示形状: {final_hidden.shape}")
    print("\n验证通过：output形状为 (seq_len, batch, 2*hidden_dim)，final_hidden形状为 (batch, 2*hidden_dim)")

形状验证
输入序列形状: torch.Size([10, 4, 8])
每个时间步拼接隐藏状态形状: torch.Size([10, 4, 32])
最终序列表示形状: torch.Size([4, 32])

验证通过：output形状为 (seq_len, batch, 2*hidden_dim)，final_hidden形状为 (batch, 2*hidden_dim)


### 5.1 
---
#### 一、负采样的核心思想
Skip-gram模型的原始目标是用中心词预测上下文词，本质是词汇表规模的多分类问题，计算softmax的开销极大。负采样（Negative Sampling）将该问题转化为**多个二分类任务**：对每一组「中心词+真实上下文词」的正样本对，采样$K$个非上下文词作为负样本，分别判断每对词是否为上下文关系，以此大幅降低计算复杂度。

---
#### 二、概率定义与符号说明
题目给定符号：
- $\boldsymbol{v}_c$：中心词$w_c$的输入词向量
- $\boldsymbol{u}_o$：真实上下文词$w_o$的输出词向量
- $\boldsymbol{u}_{n_k}$：第$k$个负样本词$w_{n_k}$的输出词向量

使用sigmoid函数计算两个词为上下文对的概率：
$$\sigma(x) = \frac{1}{1+e^{-x}}$$

1.  **正样本概率**：中心词$w_c$与上下文词$w_o$是真实上下文对的概率
    $$P(D=1 \mid w_c, w_o) = \sigma(\boldsymbol{u}_o^\top \boldsymbol{v}_c)$$

2.  **负样本概率**：中心词$w_c$与负样本$w_{n_k}$不是上下文对的概率
    $$P(D=0 \mid w_c, w_{n_k}) = 1 - \sigma(\boldsymbol{u}_{n_k}^\top \boldsymbol{v}_c) = \sigma(-\boldsymbol{u}_{n_k}^\top \boldsymbol{v}_c)$$

---
#### 三、对数似然（目标函数）
对于单个「中心词+1个正上下文词+$K$个负样本」的样本组，我们最大化其对数似然，完整的目标函数（对数似然形式）为：

$$
\boldsymbol{
\mathcal{L} = \log \sigma(\boldsymbol{u}_o^\top \boldsymbol{v}_c) + \sum_{k=1}^K \log \sigma\left(-\boldsymbol{u}_{n_k}^\top \boldsymbol{v}_c\right)
}
$$

对应的损失函数（最小化目标，为负对数似然）：
$$
J = -\log \sigma(\boldsymbol{u}_o^\top \boldsymbol{v}_c) - \sum_{k=1}^K \log \sigma\left(-\boldsymbol{u}_{n_k}^\top \boldsymbol{v}_c\right)
$$

对整个语料的所有中心词-上下文对求和，即可得到全局的总目标函数。

---
#### 四、负样本的采样方法
负样本从**噪声分布**中采样得到，标准实现采用**词频3/4次方平滑的一元语法分布**：

1.  **噪声分布公式**
    设词汇表为$V$，词$w$的语料出现频次为$\text{count}(w)$，则每个词的采样概率为：
    $$
    P_n(w) = \frac{\text{count}(w)^{3/4}}{\sum_{w_i \in V} \text{count}(w_i)^{3/4}}
    $$

2.  **采样说明**
    - 3/4次方的作用是平滑词频：降低高频词的采样权重，提升低频词的采样概率，让模型学到更均衡的词向量表示。
    - 采样时根据上述概率分布随机抽取$K$个词，通常约束负样本词不能与当前正上下文词$w_o$相同。

In [11]:
#5.2
import torch
import torch.nn.functional as F

def cbow_forward_loss(context_indices, target_indices, W, W_out):
    """
    CBOW模型前向传播 + 完整softmax交叉熵损失计算
    
    参数:
        context_indices: 上下文词索引，形状 (batch_size, context_size)
        target_indices: 中心词索引（目标），形状 (batch_size,)
        W: 输入权重矩阵，形状 (V, d)，V为词汇表大小，d为嵌入维度
        W_out: 输出权重矩阵，形状 (d, V)
    
    返回:
        loss: 批次平均交叉熵损失（标量）
    """
    # 1. 查表获取所有上下文词的嵌入向量，形状 (batch_size, context_size, d)
    context_embeds = W[context_indices]
    
    # 2. 对上下文向量取平均，得到隐藏层表示，形状 (batch_size, d)
    h = context_embeds.mean(dim=1)
    
    # 3. 计算输出层得分（logits），形状 (batch_size, V)
    logits = h @ W_out
    
    # 4. 计算完整softmax的交叉熵损失（内部包含softmax+对数运算，数值稳定）
    loss = F.cross_entropy(logits, target_indices)
    
    return loss


# ------------------- 功能验证 -------------------
if __name__ == "__main__":
    # 超参数设置
    V = 1000        # 词汇表大小
    d = 128         # 嵌入向量维度
    batch_size = 8  # 批次大小
    context_size = 4  # 每个样本的上下文词数量

    # 随机初始化权重与输入
    W = torch.randn(V, d, requires_grad=True)
    W_out = torch.randn(d, V, requires_grad=True)
    context_idx = torch.randint(0, V, (batch_size, context_size))
    target_idx = torch.randint(0, V, (batch_size,))

    # 前向计算损失
    loss = cbow_forward_loss(context_idx, target_idx, W, W_out)

    print("CBOW 前向计算验证")
    print(f"词汇表大小 V = {V}, 嵌入维度 d = {d}")
    print(f"批次大小 = {batch_size}, 上下文窗口大小 = {context_size}")
    print(f"交叉熵损失值: {loss.item():.4f}")
    print("计算完成，支持反向传播梯度回传。")

CBOW 前向计算验证
词汇表大小 V = 1000, 嵌入维度 d = 128
批次大小 = 8, 上下文窗口大小 = 4
交叉熵损失值: 17.8267
计算完成，支持反向传播梯度回传。


### 6.1
---
#### 一、已知条件
- 查询矩阵 $Q \in \mathbb{R}^{2 \times 4}$：共2个查询向量，每个向量维度 $d_k=4$
- 键矩阵 $K \in \mathbb{R}^{3 \times 4}$：共3个键向量，每个向量维度 $d_k=4$
- 值矩阵 $V \in \mathbb{R}^{3 \times 5}$：共3个值向量，每个向量维度 $d_v=5$
- 缩放因子：$\sqrt{d_k} = \sqrt{4} = 2$

缩放点积注意力的通用公式：
$$\text{Attention}(Q,K,V) = \text{softmax}\left( \frac{QK^\top}{\sqrt{d_k}} \right) V$$

---
#### 二、通用分步计算过程
##### 步骤1：计算未缩放的得分矩阵
通过查询与键的点积计算相似度，执行矩阵乘法 $Q K^\top$：
- 输入形状：$Q_{2\times4}$，$K^\top_{4\times3}$
- 输出得分矩阵形状：$\boldsymbol{Scores \in \mathbb{R}^{2 \times 3}}$
  第$i$行第$j$列元素表示第$i$个查询与第$j$个键的点积相似度：
  $$\text{Scores}_{i,j} = q_i \cdot k_j^\top$$

##### 步骤2：缩放得分矩阵
将得分矩阵除以缩放因子$\sqrt{d_k}=2$，避免点积值过大导致softmax梯度消失：
$$\text{Scores}_{\text{scaled}} = \frac{\text{Scores}}{\sqrt{d_k}} = \frac{\text{Scores}}{2}$$
矩阵形状保持$2 \times 3$不变。

##### 步骤3：softmax归一化得到注意力权重
对缩放后的得分矩阵**按行**做softmax运算，得到注意力权重矩阵$\text{Attn} \in \mathbb{R}^{2 \times 3}$：
$$\text{Attn}_{i,j} = \frac{\exp(\text{Scores}_{\text{scaled},i,j})}{\sum_{t=1}^3 \exp(\text{Scores}_{\text{scaled},i,t})}$$
每一行的权重和为1，表示对应查询对各个值的关注程度。

##### 步骤4：加权求和得到最终输出
用注意力权重对值矩阵加权求和，得到输出矩阵：
- 输入形状：$\text{Attn}_{2\times3}$，$V_{3\times5}$
- 输出矩阵形状：$\boldsymbol{Output \in \mathbb{R}^{2 \times 5}}$
  第$i$行输出为所有值向量的加权和：
  $$\text{Output}_i = \sum_{j=1}^3 \text{Attn}_{i,j} \cdot v_j$$

---
#### 三、数值计算示例
为直观演示，构造简单数值矩阵进行计算：
$$
Q = \begin{bmatrix} 1 & 0 & 1 & 0 \\ 0 & 1 & 0 & 1 \end{bmatrix}, \quad
K = \begin{bmatrix} 1 & 1 & 0 & 0 \\ 0 & 0 & 1 & 1 \\ 1 & 0 & 1 & 0 \end{bmatrix}, \quad
V = \begin{bmatrix} 1 & 2 & 3 & 4 & 5 \\ 6 & 7 & 8 & 9 & 10 \\ 11 & 12 & 13 & 14 & 15 \end{bmatrix}
$$

##### 步骤1：计算未缩放得分
$$
QK^\top = \begin{bmatrix} 1 & 0 & 1 & 0 \\ 0 & 1 & 0 & 1 \end{bmatrix}
\begin{bmatrix} 1 & 0 & 1 \\ 1 & 0 & 0 \\ 0 & 1 & 1 \\ 0 & 1 & 0 \end{bmatrix}
= \begin{bmatrix} 1 & 1 & 2 \\ 1 & 1 & 0 \end{bmatrix}
$$

##### 步骤2：缩放（除以2）
$$
\text{Scores}_{\text{scaled}} = \frac{1}{2}\begin{bmatrix} 1 & 1 & 2 \\ 1 & 1 & 0 \end{bmatrix}
= \begin{bmatrix} 0.5 & 0.5 & 1 \\ 0.5 & 0.5 & 0 \end{bmatrix}
$$

##### 步骤3：按行做softmax
- 第一行：$e^{0.5}\approx1.6487,\ e^{0.5}\approx1.6487,\ e^{1}\approx2.7183$，求和得$6.0157$
  $$\text{Attn}_{1,:} \approx [0.274,\ 0.274,\ 0.452]$$

- 第二行：$e^{0.5}\approx1.6487,\ e^{0.5}\approx1.6487,\ e^{0}=1$，求和得$4.2974$
  $$\text{Attn}_{2,:} \approx [0.384,\ 0.384,\ 0.232]$$

注意力权重矩阵：
$$
\text{Attn} \approx \begin{bmatrix} 0.274 & 0.274 & 0.452 \\ 0.384 & 0.384 & 0.232 \end{bmatrix}
$$

##### 步骤4：加权求和得到输出
$$
\text{Output} = \begin{bmatrix} 0.274 & 0.274 & 0.452 \\ 0.384 & 0.384 & 0.232 \end{bmatrix}
\begin{bmatrix} 1 & 2 & 3 & 4 & 5 \\ 6 & 7 & 8 & 9 & 10 \\ 11 & 12 & 13 & 14 & 15 \end{bmatrix}
\approx \begin{bmatrix} 6.89 & 7.89 & 8.89 & 9.89 & 10.89 \\ 5.24 & 6.24 & 7.24 & 8.24 & 9.24 \end{bmatrix}
$$

最终输出矩阵形状为$2\times5$，与维度推导结果一致。

In [10]:
#6.2
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class MultiHeadAttention(nn.Module):
    """
    多头注意力前向传播实现
    输入/输出形状: (seq_len, batch_size, d_model)
    """
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        assert d_model % num_heads == 0, "d_model 必须能被 num_heads 整除"
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads  # 每个头的Q/K/V维度，d_v = d_k
        
        # Q、K、V的全局线性投影层
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        # 多头拼接后的输出线性层
        self.W_o = nn.Linear(d_model, d_model, bias=False)
    
    def forward(self, x):
        """
        参数:
            x: 输入序列，形状 (seq_len, batch_size, d_model)
        返回:
            output: 多头注意力输出，形状与输入完全一致
        """
        seq_len, batch_size, _ = x.shape
        
        # 步骤1：线性投影得到 Q、K、V，形状均为 (seq_len, batch, d_model)
        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)
        
        # 步骤2：拆分为多个头，调整维度便于并行计算
        # 原始: (seq_len, batch, d_model) → 拆分: (seq_len, batch, num_heads, d_k)
        # 转置为 (num_heads, batch, seq_len, d_k)，每个头独立计算注意力
        Q = Q.view(seq_len, batch_size, self.num_heads, self.d_k).permute(2, 1, 0, 3)
        K = K.view(seq_len, batch_size, self.num_heads, self.d_k).permute(2, 1, 0, 3)
        V = V.view(seq_len, batch_size, self.num_heads, self.d_k).permute(2, 1, 0, 3)
        
        # 步骤3：逐头计算缩放点积注意力
        # 得分矩阵形状: (num_heads, batch, seq_len, seq_len)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        attn_weights = F.softmax(scores, dim=-1)
        # 加权求和得到每个头的输出，形状 (num_heads, batch, seq_len, d_k)
        attn_output = torch.matmul(attn_weights, V)
        
        # 步骤4：拼接所有头的输出
        # 维度还原为 (seq_len, batch, num_heads, d_k)
        attn_output = attn_output.permute(2, 1, 0, 3).contiguous()
        # 最后一维拼接，恢复为 d_model 维度
        attn_concat = attn_output.view(seq_len, batch_size, self.d_model)
        
        # 步骤5：经过最终输出线性层
        output = self.W_o(attn_concat)
        
        return output


# ------------------- 形状验证（题目指定参数） -------------------
if __name__ == "__main__":
    # 题目指定参数
    d_model = 4
    num_heads = 2
    seq_len = 5
    batch_size = 3
    
    # 随机生成输入
    x = torch.randn(seq_len, batch_size, d_model)
    
    # 初始化并前向传播
    mha = MultiHeadAttention(d_model, num_heads)
    output = mha(x)
    
    print("多头注意力形状验证")
    print(f"输入形状: {x.shape}  (seq_len, batch, d_model)")
    print(f"输出形状: {output.shape}  (seq_len, batch, d_model)")
    print("输出与输入形状一致，满足题目要求。")

多头注意力形状验证
输入形状: torch.Size([5, 3, 4])  (seq_len, batch, d_model)
输出形状: torch.Size([5, 3, 4])  (seq_len, batch, d_model)
输出与输入形状一致，满足题目要求。
